In [236]:
!pip install rdflib >> /dev/null
!pip install pandas >> /dev/null
!pip install pydicom >> /dev/null
!pip install rdflib_jsonld >> /dev/null

El sistema no puede encontrar la ruta especificada.
El sistema no puede encontrar la ruta especificada.
El sistema no puede encontrar la ruta especificada.
El sistema no puede encontrar la ruta especificada.


In [237]:
import os

# Get the directory of the current script
base_dir = os.getcwd()
try:
    if folder:
        base_dir = os.path.join(base_dir, folder)
except:
    pass

In [238]:
from rdflib import * 
import uuid
from hashlib import sha256
import json

In [ ]:
tbox = Namespace('http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#')
abox = Namespace('http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#')

# W3C Standard Vocabularies
dcat = Namespace('http://www.w3.org/ns/dcat#')
csvw = Namespace('http://www.w3.org/ns/csvw#')
dcterms = Namespace('http://purl.org/dc/terms/')
schema = Namespace('http://schema.org/')
odrl = Namespace('http://www.w3.org/ns/odrl/2/')
dqv = Namespace('http://www.w3.org/ns/dqv#')
prov = Namespace('http://www.w3.org/ns/prov#')

In [ ]:
class Federator:
    """
    Federator class is responsible for federating the data from different sources.
    Generates standards-compliant RDF using ODRL, Schema.org, and CSVW.
    """

    def __init__(self, ds, SDM):
        self.ds = ds
        self.sdm = SDM 

    def check_dp_existance(self):
        """Check if the data product already exists in the SDM."""
        if self.sdm.value(predicate=RDF.type, subject=abox[self.ds]):
            return True 
        else:
            return False
        
    def generate_uri_id(self):
        """Generate a unique UUID for URIs."""
        return str(uuid.uuid4())
        
    def add_mappings(self, mappings):
        """
        Add schema mappings to the data contract.
        Mappings are from physical attributes (ab:ID) to semantic features (ab:Subject).
        Creates CSVW propertyUrl mappings where possible.
        """
        # Create data contract (both custom and ODRL standard)
        contract_uri = abox[f'dc_{self.ds}']
        self.sdm.add((contract_uri, RDF.type, tbox.DataContract))
        self.sdm.add((contract_uri, RDF.type, odrl.Agreement))
        self.sdm.add((abox[self.ds], tbox.hasDC, contract_uri))
        
        # Add mappings
        for physical_attr, semantic_feature in mappings.items():
            # Generate mapping UUID
            mapping_uuid = self.generate_uri_id()
            mapping_uri = abox[mapping_uuid]
            
            # Create schema mapping
            self.sdm.add((mapping_uri, RDF.type, tbox.SchemaMapping))
            self.sdm.add((contract_uri, tbox.hasMapping, mapping_uri))
            
            # Map from physical attribute to semantic feature
            self.sdm.add((mapping_uri, tbox.mfrom, abox[physical_attr]))
            self.sdm.add((mapping_uri, tbox.mto, abox[semantic_feature]))
            
            # Add CSVW propertyUrl mapping to the attribute itself
            # This links the column to its semantic meaning via Schema.org
            attr_uri = abox[physical_attr]
            
            # Map common column names to Schema.org properties
            schema_mapping = self._get_schema_mapping(physical_attr, semantic_feature)
            if schema_mapping:
                self.sdm.add((attr_uri, csvw.propertyUrl, schema_mapping))
                
                # Also add it as tb:semanticFeature for clarity
                self.sdm.add((attr_uri, tbox.semanticFeature, Literal(semantic_feature)))
        
        return self.sdm
    
    def _get_schema_mapping(self, physical_attr, semantic_feature):
        """
        Map physical attributes and semantic features to Schema.org properties.
        Returns URIRef of Schema.org property or None.
        """
        # Common mappings
        mappings = {
            'ID': schema.identifier,
            'Subject': schema.identifier,
            'Age': schema.age,
            'Age_at_scan_years': schema.age,
            'Gender': schema.gender,
            'ts': schema.dateCreated,
            'timestamp': schema.dateCreated,
            'Survival': schema.duration,
            'Survival_from_surgery_days_UPDATED': schema.duration,
            'KPS': schema.medicalCondition,
            'MGMT': schema.medicalCondition,
            'IDH1': schema.medicalCondition,
        }
        
        # Check both physical attribute and semantic feature
        if physical_attr in mappings:
            return mappings[physical_attr]
        elif semantic_feature in mappings:
            return mappings[semantic_feature]
        else:
            # Default to schema:propertyValue for unmapped attributes
            return schema.propertyValue
            
    def add_policies(self, policies):
        """
        Add agreed policies to the data contract.
        Policies should already exist in the SDM as ODRL policies.
        """
        contract_uri = abox[f'dc_{self.ds}']
        
        # Add agreed policies
        for policy in policies:
            self.sdm.add((contract_uri, tbox.hasPolicy, abox[policy]))
            # Also link via ODRL terms
            self.sdm.add((contract_uri, odrl.permission, abox[policy]))
            
        return self.sdm

Run the Federator

Load SDM

In [241]:
sdm = Graph().parse(os.path.join(base_dir, '../../FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')

Data Product Metadata

In [242]:
import json 
try:
    if dp_meta_path:
        dp_meta = json.load(open(dp_meta_path))
        dataset = dp_meta['name']
        mappings = dp_meta['mappings']
        policies = dp_meta['policies']
except:
    pass

In [243]:
federator = Federator(dataset, sdm)

In [244]:
print(federator.check_dp_existance())   

True


In [245]:
contract = federator.add_mappings(mappings)

In [246]:
contract = federator.add_policies(policies)

Save Contract that includes the specified Integration

In [247]:
sdm = Graph().parse(os.path.join(base_dir, '../../FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')
sdm += contract
sdm.serialize(destination=os.path.join(base_dir, '../../FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')

<Graph identifier=N73cf817ca67c4569917b6e89d6fa7955 (<class 'rdflib.graph.Graph'>)>